In [1]:
import keras
keras.__version__

Using TensorFlow backend.


'2.2.4'

##  مقدمة للشبكات الترابطية convnet:

أولاً، فلنأخذ مثالاً عملياً بسيط لهذه الشبكات . ولنستخدم الشبكة الترابطية هنا لتصنيف خانات MNIST وهي المهمة التي تم القيام بها مسبقاً بوساطة الشبكة العصبونية الكثيفة مسبقاً (حيث كانت الدقة حينها 97.8%) . أما الآن فحتى ضمن نموذج بسيط للشبكة الترابطية فسنحصل على أداء أفضل من الشبكة الكثيفة.


الأسطر الستة التالية ستظهر لك شبكة ترابطية أساسية من خلال البرنامج المعبر عنها. هي شبكة من تتالي طبقات `Conv2D` و طبقات `MaxPooling2D` وسنرى بالتالي ما تقوم به بالتحديد . ومن الأهمية بمكان ملاحظة أن دخل هذه الشبكات هو مصفوفة من الشكل (ارتفاع الصورة، عرض الصورة ، قنوات الصورة) وهذا لا يتضمن حجم الحزمة المعالجة . وبحالتنا سنعاير الشبكة لمعالجة دخل من القياس `(28,28,1)` وهذه هي صيغة صور MNIST ونقوم بهذا عبر تمرير المتغير التالي للطيقة الأولى بالشبكة: `input_shape=(28,28,1)`


In [2]:
from keras import layers
from keras import models

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

لنظهر البنية للشبكة التي تم بناؤها الآن عبر تنفيذ التالي:

In [3]:
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 64)        18496     
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 64)          0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 3, 3, 64)          36928     
Total params: 55,744
Trainable params: 55,744
Non-trainable params: 0
_________________________________________________________________


يمكنك أن ترى أن خرج كل طبقات `Conv2D` , `MaxPooling2D` هو مصفوفة ثلاثية الأبعاد من الشكل `(طول وعرض وقنوات)` .
وأن أبعاد الطول والعرض تكيل للتقلص مع ازدياد ترتيب الطبقة  وعدد القنوات يتم التحكم به من خلال أول نتغير يمرر لطبقات `Conv2D` مثلاً 64 أو 32
.

الخطوة التالية ستكون بإدخال الخرج الحالي المكون من مصفوفات بحجم `(3,3,64)` لمصنف مترابط بشكل كثيف كشبكات مشابهة لماتم العمل معه مسبقاً ، وهي سلسلة من طبقات `Dense` . وهذه المصنفات تعالج الأشعة ، والتي تمثل كمصفوفات ببعد واحد ولكن الخرج الحالي هو بثلاثة أبعاد . لذلك أولاً، سيجب علينا تسوية الخرج ثلاثي الأبعاد لأحادي البعد ومن ثم ستتم إضافة طبقات `Dense` لأعلى النموذج الحالي

In [4]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))

  بما أننا سنبني مصنفاً لعشر فئات فسنجعل طبقة الخرج ب 10  عصبونات وتابع تفعيل `softmax` وهذا ما ستبدو عليه الشبكة النهائية:

In [5]:
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 64)        18496     
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 64)          0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 3, 3, 64)          36928     
_________________________________________________________________
flatten_1 (Flatten)          (None, 576)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 64)                36928     
__________

كما ترى فإن الخرج من القياس `(3,3,64)` سيتم تسويته لأشعة من الشكل `(576,)` قبل إدخاله للطبقات الكثيفة .

والآن سنقوم بتدريب شبكة convnet على خانات MNIST . وسنعيد استخدام أجزاء من البرنامج المستخدم بالتدريب للحالة عند استخدام الشبكات الكثيفة فقط كما ورد مسبقاً.


In [6]:
from keras.datasets import mnist
from keras.utils import to_categorical

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype('float32') / 255

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

In [7]:
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=5, batch_size=64)

Epoch 1/5
60000/60000 [==============================] - 75s 1ms/step - loss: 0.1674 - acc: 0.9478
Epoch 2/5
60000/60000 [==============================] - 70s 1ms/step - loss: 0.0488 - acc: 0.9847
Epoch 3/5
60000/60000 [==============================] - 70s 1ms/step - loss: 0.0324 - acc: 0.9899
Epoch 4/5
60000/60000 [==============================] - 72s 1ms/step - loss: 0.0246 - acc: 0.9925
Epoch 5/5
60000/60000 [==============================] - 70s 1ms/step - loss: 0.0200 - acc: 0.9938


فلنقم بتقييم النموذج على بيانات الاحتبار

In [8]:
test_loss, test_acc = model.evaluate(test_images, test_labels)

10000/10000 [==============================] - 4s 403us/step


In [9]:
test_acc

0.9904

في حين أن شبكتنا المترابطة بشكل كثيف في أول مثال كانت دقتها بأعلى مستوى لها في 97.8% فإن الشبكة الأولية المعتمدة على convnet حققت دقة اختبار بمقدار 99.3% ، حيث قلصنا الخطأ بمقدار 68% نسبياً . وهذا تحسين معتبر.